In [56]:
import os
from shutil import move

import numpy as np
import torch
import torch.nn.functional as F
import torch.optim as optim
from PIL import Image
from kagglehub import kagglehub
from torch import nn
from torch.utils.data import DataLoader
from torch.utils.data import WeightedRandomSampler
from torch.utils.data import random_split
from torch.utils.tensorboard import SummaryWriter
from torchvision import datasets, transforms
from tqdm import tqdm
from skopt.space import Real, Integer, Categorical
from skopt import gp_minimize



In [57]:
if torch.cuda.is_available():
    print("CUDA is available!")
    print(f"Device name: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA is not available.")


CUDA is available!
Device name: NVIDIA GeForce RTX 4060 Laptop GPU


In [58]:
# Generalized permanent dataset path
home_dir = os.path.expanduser("~")  # Gets the user's home directory
permanent_path = os.path.join(home_dir, "datasets")

# Download dataset if it doesn't exist
if not os.path.exists(permanent_path):
    print("Downloading dataset...")
    temp_path = kagglehub.dataset_download("subhaditya/fer2013plus")  # Temporary download path
    print("Moving dataset to permanent location...")
    move(temp_path, permanent_path)
    print("Dataset moved to:", permanent_path)
else:
    print("Dataset already exists at:", permanent_path)

# Identify train and test directories dynamically
train_dir = os.path.join(permanent_path, "fer2013plus", "fer2013", "train")
test_dir = os.path.join(permanent_path, "fer2013plus", "fer2013", "test")

Dataset already exists at: /home/omarhammad/datasets


In [59]:
# Paths for augmented datasets
augmented_train_dir = os.path.join(permanent_path, "fer2013plus/fer2013/train_augmented")
augmented_test_dir = os.path.join(permanent_path, "fer2013plus/fer2013/test_augmented")

# Define augmentations for train and processing for test
train_augment_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),  # Ensure image is grayscale (1 channel)
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomCrop(48, padding=4),
    transforms.ToTensor(),  # Convert to tensor at the end
    transforms.Normalize((0.5,), (0.5,))  # Normalize after tensor conversion
])

test_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),  # Ensure image is grayscale (1 channel)
    transforms.ToTensor(),  # Convert to tensor at the beginning
    transforms.Normalize((0.5,), (0.5,))  # Normalize during loading
])

# Check if the augmented datasets exist
if not os.path.exists(augmented_train_dir) or not os.path.exists(augmented_test_dir):
    print("Augmented datasets not found. Creating augmented datasets...")

    # Use GPU for processing if available
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Create and save the augmented train dataset
    print("Processing train dataset...")
    for label in os.listdir(train_dir):
        label_dir = os.path.join(train_dir, label)
        save_dir = os.path.join(augmented_train_dir, label)
        os.makedirs(save_dir, exist_ok=True)
        for img_name in tqdm(os.listdir(label_dir), desc=f"Processing {label}"):
            img_path = os.path.join(label_dir, img_name)
            img = Image.open(img_path).convert("L")  # Convert to grayscale (1 channel)

            # Apply augmentations to the PIL Image
            for i in range(5):  # Generate 5 augmented images per original
                augmented_img = train_augment_transform(img)  # Apply transforms directly to the PIL Image
                augmented_img_pil = transforms.ToPILImage()(
                    augmented_img)  # Convert tensor back to PIL Image for saving
                augmented_img_pil.save(os.path.join(save_dir, f"{os.path.splitext(img_name)[0]}_aug_{i}.png"))

    # Process and save the test dataset
    print("Processing test dataset...")
    for label in os.listdir(test_dir):
        label_dir = os.path.join(test_dir, label)
        save_dir = os.path.join(augmented_test_dir, label)
        os.makedirs(save_dir, exist_ok=True)
        for img_name in tqdm(os.listdir(label_dir), desc=f"Processing {label}"):
            img_path = os.path.join(label_dir, img_name)
            img = Image.open(img_path).convert("L")  # Convert to grayscale (1 channel)

            # Apply test transforms directly
            processed_img = test_transform(img)
            processed_img_pil = transforms.ToPILImage()(processed_img)  # Convert tensor back to PIL Image for saving
            processed_img_pil.save(os.path.join(save_dir, img_name))
else:
    print("Augmented datasets already exist. Skipping processing.")

# Reload processed datasets with normalization applied during loading
train_dataset = datasets.ImageFolder(augmented_train_dir, transform=train_augment_transform)
test_dataset = datasets.ImageFolder(augmented_test_dir, transform=test_transform)


Augmented datasets already exist. Skipping processing.


In [60]:
# Dataset sizes
print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

Train dataset size: 141930
Test dataset size: 7099


In [61]:
def create_dataloaders(train_dataset, test_dataset, batch_size, val_split=0.2):
    train_size = int((1 - val_split) * len(train_dataset))
    val_size = len(train_dataset) - train_size
    generator = torch.Generator().manual_seed(42)  # For reproducibility
    train_subset, val_subset = random_split(train_dataset, [train_size, val_size], generator=generator)

    if not hasattr(train_dataset, 'targets'):
        raise AttributeError("train_dataset must have 'targets' attribute for class balancing.")

    train_labels = [train_dataset.targets[i] for i in train_subset.indices]
    class_counts = np.bincount(train_labels)
    class_weights = 1. / np.sqrt(class_counts + 1e-6)
    class_weights = torch.tensor(class_weights, dtype=torch.float32)

    sample_weights = [class_weights[label] for label in train_labels]
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

    train_loader = DataLoader(train_subset, batch_size=batch_size, sampler=sampler)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader


In [62]:

def train_model(model, train_loader, val_loader, optimizer, criterion, num_epochs=100, print_every=100, patience=5):
    """
    Trains a given model with early stopping and logs metrics to TensorBoard.

    Args:
        model: The model to train.
        train_loader: DataLoader for the training dataset.
        val_loader: DataLoader for the validation dataset.
        optimizer: Optimizer for training (e.g., Adam, SGD).
        criterion: Loss function (e.g., CrossEntropyLoss).
        num_epochs: Number of training epochs.
        print_every: Frequency (in batches) to print training loss.
        patience: Number of epochs to wait for validation loss improvement before stopping.
    """
    # Initialize TensorBoard writer
    writer = SummaryWriter(log_dir="../logs")

    # Set device (CUDA if available, otherwise CPU)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)

    best_val_loss = float('inf')
    patience_counter = 0

    for epoch in range(num_epochs):
        model.train()  # Set the model to training mode
        running_loss = 0.0

        for i, (inputs, labels) in enumerate(train_loader):
            # Move data to GPU (if available)
            inputs, labels = inputs.to(device), labels.to(device)

            # Zero the parameter gradients
            optimizer.zero_grad()

            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            # Backward pass
            loss.backward()
            optimizer.step()

            # Accumulate the loss
            running_loss += loss.item()

            # Log batch loss to TensorBoard
            if (i + 1) % print_every == 0:
                avg_loss = running_loss / print_every
                writer.add_scalar("Training Loss", avg_loss, epoch * len(train_loader) + i)
                running_loss = 0.0

        # Validation phase
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
        val_loss /= len(val_loader)

        # Log validation loss to TensorBoard
        writer.add_scalar("Validation Loss", val_loss, epoch)

        print(f"Epoch [{epoch + 1}/{num_epochs}], Validation Loss: {val_loss:.4f}")

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping triggered at epoch {epoch + 1}")
                break

    # Close TensorBoard writer
    writer.close()


In [63]:
def evaluate_model(model, data_loader):
    """
    Evaluates the model's performance on a given dataset.

    Args:
        model: The trained model to evaluate.
        data_loader: DataLoader for the dataset to evaluate.

    Returns:
        float: Accuracy of the model on the dataset.
    """
    # Set the device (CUDA if available, otherwise CPU)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Move model to the device
    model = model.to(device)

    model.eval()  # Set model to evaluation mode
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in data_loader:
            # Move data to the same device as model
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = correct / total
    return accuracy


In [64]:
class CNN(nn.Module):
    def __init__(self, dropout_rate=0.3):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)
        self.dropout = nn.Dropout(dropout_rate)
        self.fc1 = nn.Linear(128 * 6 * 6, 256)
        self.fc2 = nn.Linear(256, 8)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.view(-1, 128 * 6 * 6)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x


In [65]:
search_space = [
    Real(1e-6, 1e-2, name='learning_rate', prior='log-uniform'),
    Integer(16, 64, name='batch_size'),
    Categorical(['Adam', 'SGD', 'RMSprop', 'AdamW'], name='optimizer'),
    Real(0.1, 0.5, name='dropout_rate'),
    Real(1e-5, 1e-2, name='weight_decay', prior='log-uniform'),
    Integer(3, 10, name='patience')  # Early stopping patience
]


In [66]:
def objective(params):
    learning_rate, batch_size, optimizer_type, dropout_rate, weight_decay, patience = params
    batch_size = int(batch_size)
    patience = int(patience)

    # Set the device (use GPU if available, otherwise CPU)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Update data loaders
    train_loader, val_loader, _ = create_dataloaders(train_dataset, test_dataset, batch_size)

    # Define model, criterion, and optimizer
    model = CNN(dropout_rate=dropout_rate).to(device)  # Pass dropout_rate to CNN
    criterion = nn.CrossEntropyLoss()

    # Use the correct optimizer with weight decay
    optimizer = getattr(optim, optimizer_type)(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    # Early stopping logic
    best_val_loss = float('inf')
    patience_counter = 0

    print(
        f"Starting training with params: lr={learning_rate}, batch_size={batch_size}, optimizer={optimizer_type}, dropout_rate={dropout_rate}, weight_decay={weight_decay}, patience={patience}, device={device}")

    epochs = 100
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        # Training loop
        for batch_idx, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)  # Move data to GPU
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

            # Print batch progress
            if (batch_idx + 1) % 100 == 0:  # Every 100 batches
                print(f"Epoch [{epoch + 1}/{epochs}], Batch [{batch_idx + 1}/{len(train_loader)}], Loss: {loss.item():.4f}")

        # Validation loss
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)  # Move data to GPU
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
        val_loss /= len(val_loader)

        print(f"Epoch [{epoch + 1}/{epochs}], Validation Loss: {val_loss:.4f}")

        # Early stopping
        if val_loss < best_val_loss:
            print(f"Validation loss improved from {best_val_loss:.4f} to {val_loss:.4f}")
            best_val_loss = val_loss
            patience_counter = 0  # Reset patience counter if improvement
        else:
            patience_counter += 1
            print(f"No improvement in validation loss. Patience counter: {patience_counter}/{patience}")

            if patience_counter >= patience:
                print(f"Early stopping triggered after {epoch + 1} epochs")
                break

    # Calculate validation accuracy
    total, correct = 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)  # Move data to GPU
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_accuracy = correct / total
    print(f"Final Validation Accuracy: {val_accuracy:.4f}")
    return -val_accuracy


In [67]:

result = gp_minimize(
    func=objective,
    dimensions=search_space,
    n_calls=10,
    random_state=42
)

# Define the parameter names in the same order as in the search space
param_names = ['learning_rate', 'batch_size', 'optimizer', 'dropout_rate', 'weight_decay', 'patience']

# Print each parameter with its name
print("Best parameters:")
for name, value in zip(param_names, result.x):
    print(f"{name} = {value}")

# Print the best validation accuracy
print(f"Best validation accuracy: {-result.fun:.4f}")


Starting training with params: lr=0.001535224694197351, batch_size=25, optimizer=AdamW, dropout_rate=0.33874006317859484, weight_decay=0.00021751953118777657, patience=4, device=cuda
Epoch [1/100], Batch [100/4542], Loss: 1.8560
Epoch [1/100], Batch [200/4542], Loss: 1.9112
Epoch [1/100], Batch [300/4542], Loss: 2.0006
Epoch [1/100], Batch [400/4542], Loss: 1.8513
Epoch [1/100], Batch [500/4542], Loss: 1.9355
Epoch [1/100], Batch [600/4542], Loss: 1.8869
Epoch [1/100], Batch [700/4542], Loss: 1.9515
Epoch [1/100], Batch [800/4542], Loss: 1.8387
Epoch [1/100], Batch [900/4542], Loss: 1.8532
Epoch [1/100], Batch [1000/4542], Loss: 1.8725
Epoch [1/100], Batch [1100/4542], Loss: 2.0315
Epoch [1/100], Batch [1200/4542], Loss: 1.7279
Epoch [1/100], Batch [1300/4542], Loss: 2.0295
Epoch [1/100], Batch [1400/4542], Loss: 1.8795
Epoch [1/100], Batch [1500/4542], Loss: 1.9514
Epoch [1/100], Batch [1600/4542], Loss: 1.9145
Epoch [1/100], Batch [1700/4542], Loss: 1.8431
Epoch [1/100], Batch [1800/

In [68]:
# Extract best parameters from tuning results
best_learning_rate = result.x[0]
best_batch_size = int(result.x[1])
best_optimizer_type = result.x[2]
best_dropout_rate = result.x[3]
best_weight_decay = result.x[4]
best_patience = int(result.x[5])

# Update data loaders with the best batch size
train_loader, val_loader, test_loader = create_dataloaders(train_dataset, test_dataset, best_batch_size)

# Define the model, criterion, and optimizer
best_cnn_model = CNN(dropout_rate=best_dropout_rate)
criterion = nn.CrossEntropyLoss()

# Use the best optimizer type with weight decay
optimizer = getattr(torch.optim, best_optimizer_type)(
    best_cnn_model.parameters(),
    lr=best_learning_rate,
    weight_decay=best_weight_decay
)

# Train the model with early stopping
train_model(
    model=best_cnn_model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    patience=best_patience
)

Epoch [1/100], Validation Loss: 1.5733
Epoch [2/100], Validation Loss: 1.5105
Epoch [3/100], Validation Loss: 1.3994
Epoch [4/100], Validation Loss: 1.3304
Epoch [5/100], Validation Loss: 1.2888
Epoch [6/100], Validation Loss: 1.2650
Epoch [7/100], Validation Loss: 1.2270
Epoch [8/100], Validation Loss: 1.1779
Epoch [9/100], Validation Loss: 1.1690
Epoch [10/100], Validation Loss: 1.1094
Epoch [11/100], Validation Loss: 1.0942
Epoch [12/100], Validation Loss: 1.0613
Epoch [13/100], Validation Loss: 1.0412
Epoch [14/100], Validation Loss: 1.0329
Epoch [15/100], Validation Loss: 1.0275
Epoch [16/100], Validation Loss: 1.0116
Epoch [17/100], Validation Loss: 0.9760
Epoch [18/100], Validation Loss: 0.9601
Epoch [19/100], Validation Loss: 0.9532
Epoch [20/100], Validation Loss: 0.9437
Epoch [21/100], Validation Loss: 0.9509
Epoch [22/100], Validation Loss: 0.9345
Epoch [23/100], Validation Loss: 0.9246
Epoch [24/100], Validation Loss: 0.8863
Epoch [25/100], Validation Loss: 0.8641
Epoch [26

In [69]:
train_accuracy = evaluate_model(best_cnn_model, train_loader)
print(f"Train Accuracy: {train_accuracy:.4f}")

Train Accuracy: 0.8769


In [70]:
test_accuracy = evaluate_model(best_cnn_model, test_loader)
print(f"Test Accuracy: {test_accuracy:.4f}")

Test Accuracy: 0.7328


In [71]:
# Save the model's state_dict
torch.save(best_cnn_model.state_dict(), 'best_cnn_model.pth')